# Example 6-10: Velocity Change for a Rendezvous Using Noncoplanar Orbits
### _Fundamentals of Astrodynamics and Applications_, 5th Ed., 2022, pp. 373-375

This notebook demonstrates the process to find the change in velocity for a rendezvous from noncoplanar orbits.

## Install and Import Libraries
---

First, install `valladopy` if it doesn't already exist in your environment:

In [1]:
!pip install -r ../valladopy_version.txt

Import the relevant `valladopy` modules:

In [2]:
import numpy as np
import valladopy.constants as const
from valladopy.astro.maneuver.transfer import rendezvous_noncoplanar, utils

## Problem Definition
---

GIVEN:&ensp; The following circular, noncoplanar orbits for the interceptor and target<br>
FIND: &emsp;$\Delta{v}$

**Interceptor** &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; **Target** <br>
$a$ = 7143.51 km &emsp;&emsp;&emsp;&emsp;&ensp; $a$ = 42,159.4855 km <br>
$i$ = 28.5° &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&ensp; $i$ = 0° <br>
$\Omega$ = 45° &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&ensp; $\lambda_{true}$ = 200° <br>
$u$ = 15°

In [3]:
# Interceptor
a_interceptor = 7143.51              # km
i_interceptor = np.radians(28.5)     # rad
raan_interceptor = np.radians(45)    # rad
arglat_interceptor = np.radians(15)  # rad

# Target
a_target = 42159.4855                # km
i_target = 0                         # rad
truelon_target = np.radians(200)     # rad

## Solution
---

**Algorithm 46** summarizes the noncoplanar phasing process.

First, find the desired lead angle, $\alpha_L$ for the final transfer, and some initial quantities:

$$
\begin{aligned}
\omega_{tgt} &= \sqrt{\frac{\mu}{a_{tgt}^3}}, \ \ \omega_{int} = \sqrt{\frac{\mu}{a_{int}^3}} \\
\\
a_{trans} &= \frac{a_{initial} + a_{final}}{2} \\
\\
\tau_{trans} &= \pi \ \sqrt{\frac{a_{trans}^3}{\mu}} \\
\\
\alpha_L &= \omega_{tgt} \ \tau_{trans}
\end{aligned}
$$

Find the angle the interceptor must travel until reaching a node:
$$
\Delta\vartheta_{int} = 180° \text{or} \ 360° - u_{int}
$$

This allows you to calculate the amount of time required for such a movement:

$$
\Delta{t_{node}} = \frac{\Delta\vartheta_{int}}{\omega_{int}}
$$

The angular distance the target has traveled from its initial location is:

$$
\lambda_{true_1} = \lambda_{true_0} + \omega_{tgt} \ \Delta{t_{node}}
$$

Find $\lambda_{true}$ for the interceptor at $t_1$ with:

$$
\lambda_{true_{int1}} = \Omega + \pi
$$

To get the new phase angle:

$$
\vartheta_{new} = \lambda_{true_{int1}} - \lambda_{true_{tgt1}}
$$

And consequently the new lead angle:

$$
\alpha_{new} = \pi + \vartheta_{new}
$$

Determine the period of the phasing orbit and corresponding semimajor axis with:

$$
\begin{aligned}
\mathcal{P_{phase}} &= \frac{\alpha_{new} - \alpha_L + 2 \ \pi\ k_{tgt}}{\omega_{tgt}} \\
a_{phase} &= \left(
\mu \left( \frac{\mathcal{P_{phase}}}{k_{int} \ 2 \ \pi} \right) ^2
\right) ^ {1/3}
\end{aligned}
$$

After calculating $a_{phase}$, verify that $a_{int} < a_{phase} < a_{tgt}$ to ensure there is no wasted change in velocity.

Finally, calculate the velocities required for each maneuver. The initial velocities can be calculated with:

$$
v_{int} = \sqrt{\frac{\mu}{a_{int}}}, \ \ v_{tgt} = \sqrt{\frac{\mu}{a_{tgt}}}
$$

The delta velocities to: <br> 
(1) Enter the phasing orbit ($\Delta{v_{phase}}$), <br>
(2) Enter the transfer orbit ($\Delta{v_{trans1}}$), and <br>
(3) Complete the transfer to the target orbit ($\Delta{v_{trans2}}$)

can be calculated with:

$$
\begin{aligned}
\Delta{v_{phase}} &= \left|
\sqrt{\frac{2\mu}{a_{int}} - \frac{\mu}{a_{phase}}} -
\sqrt{\frac{\mu}{a_{int}}}
\right|
\\
\Delta{v_{trans_1}} &= \left|
\sqrt{\frac{2\mu}{a_{int}} - \frac{\mu}{a_{trans}}} -
\sqrt{\frac{2\mu}{a_{int}} - \frac{\mu}{a_{phase}}}
\right|
\\
\Delta{v_{trans_2}} &= \sqrt{
\left( \frac{2\mu}{a_{tgt}} - \frac{\mu}{a_{trans}} \right) +
\left( \frac{\mu}{a_{tgt}} \right) -
2 \ \sqrt{\frac{2\mu}{a_{tgt}} - \frac{\mu}{a_{trans}}}
\sqrt{\frac{u}{a_{tgt}}} \cos(\Delta{i})
}
\end{aligned}
$$

Lastly, the total time is:

$$
\begin{aligned}
\tau_{total} &= \tau_{phase} + \tau_{trans} + \Delta{t_{node}} \\ \\
\tau_{total} &= 2 \pi \sqrt{\frac{a_{phase}^3}{\mu}} + \tau_{trans} + \Delta{t_{node}}
\end{aligned}
$$

Use the `rendezvous_noncoplanar` routine to determine all these parameters, assuming no additional target revolutions ($k_{tgt}=0$) and one revolution of the interceptor orbit ($k_{int}=1)$:

In [4]:
# Define number of revolutions
k_tgt = 0
k_int = 1

# Determine interceptor angle to reach node and corresponding time
phase_i = np.pi - arglat_interceptor
t_node = phase_i / utils.angular_velocity(a_interceptor)

# Calculate the noncoplanar rendezvous parameters
t_trans, t_phase, dv_phase, dv_trans1, dv_trans2, a_phase = rendezvous_noncoplanar(
    phase_i,
    a_interceptor,
    a_target,
    k_int,
    k_tgt,
    raan_interceptor,
    truelon_target,
    deltai=(i_target - i_interceptor)
)

print(f'SMA phase:\t{a_phase:.3f}\tkm\n')
print(f'Δv phase:\t{dv_phase:.6f}\tkm/s')
print(f'Δv trans1:\t{dv_trans1:.6f}\tkm/s')
print(f'Δv trans2:\t{dv_trans2:.6f}\tkm/s')
print(f'Total Δv:\t{(dv_phase + dv_trans1 + dv_trans2):.6f}\tkm/s\n')
print(f'Nodal time:\t{t_node / const.MIN2SEC:.3f}\t\tmin')
print(f'Transfer time:\t{t_trans / const.MIN2SEC:.3f}\t\tmin')
print(f'Phase time:\t{t_phase / const.MIN2SEC:.3f}\t\tmin')
print(f'Total time:\t{(t_node + t_trans + t_phase) / const.MIN2SEC:.3f}\t\tmin')

SMA phase:	19473.304	km

Δv phase:	2.076273	km/s
Δv trans1:	0.222608	km/s
Δv trans2:	1.802449	km/s
Total Δv:	4.101330	km/s

Nodal time:	45.900		min
Transfer time:	320.992		min
Phase time:	450.733		min
Total time:	817.625		min
